# Project FORESIGHT — Risk Scoring

In [ ]:
import pandas as pd
import numpy as np
import os

DATA_DIR = "../data"
inventory = pd.read_pickle(os.path.join(DATA_DIR,"inventory_clean.pkl"))
weekly = pd.read_pickle(os.path.join(DATA_DIR,"weekly_baseline.pkl"))

In [ ]:
# Recent average weekly demand
weekly["week_start"] = pd.to_datetime(weekly["week_start"])
latest = weekly["week_start"].max()
recent = weekly[weekly["week_start"] >= latest-pd.Timedelta(weeks=8)]

demand = recent.groupby("sku_id")["units_sold"].mean().reset_index(name="avg_weekly_demand")
risk = inventory.merge(demand,on="sku_id",how="left")
risk["avg_weekly_demand"] = risk["avg_weekly_demand"].fillna(0)

In [ ]:
# Weeks of inventory cover
risk["weeks_of_cover"] = np.where(
    risk["avg_weekly_demand"] > 0,
    risk["stock_on_hand"] / risk["avg_weekly_demand"],
    np.inf
)

risk["stockout_risk"] = risk["stock_on_hand"] <= risk["safety_stock"]
risk["reorder_risk"] = risk["stock_on_hand"] <= risk["reorder_point"]
risk["overstock_risk"] = risk["weeks_of_cover"] > 8

In [ ]:
def decision(row):
    if row["stockout_risk"]:
        return "Reorder Now"
    if row["overstock_risk"]:
        return "Markdown / Clear"
    if row["reorder_risk"]:
        return "Watch"
    return "Healthy"

risk["decision"] = risk.apply(decision,axis=1)

display(risk[["store_id","sku_id","stock_on_hand","reorder_point",
              "safety_stock","avg_weekly_demand","weeks_of_cover","decision"]].head(20))

In [ ]:
display(risk["decision"].value_counts().to_frame("count"))

risk.to_pickle(os.path.join(DATA_DIR,"inventory_risk.pkl"))
risk.to_csv(os.path.join(DATA_DIR,"inventory_risk.csv"),index=False)
print("Risk results saved.")